# Intelligent Machine Learning examples for chromatography

**HPLC 2026 — Introduction to Artificial Intelligence for Liquid-phase Separations**

**Prepared for the course by Dr. Tijmen S. Bos.**  
This notebook is part of the course **Introduction to Artificial Intelligence for Liquid-phase Separations** and is intended as browser-based teaching material for the HPLC 2026 course.

**Company affiliation.**  
This material is part of **Innovative Data Evaluation And Separations (IDEAS)**, Dr. Tijmen S. Bos's company for data evaluation and separations education/consultancy. Website: **bos-ideas.com**.

**Ownership and use.**  
The educational content in this notebook belongs to **Dr. Tijmen S. Bos / Innovative Data Evaluation And Separations (IDEAS)**, unless a different source is explicitly indicated. The examples are provided for course participation, demonstration, and educational discussion.


**Non-liability and educational-use note.**  
This notebook is provided for course teaching, demonstration, and educational discussion only. The examples use simplified and/or synthetic data and are not validated analytical methods. They should not be used as the sole basis for experimental, regulatory, quality-control, safety, commercial, medical, legal, or financial decisions. Dr. Tijmen S. Bos and Innovative Data Evaluation And Separations (IDEAS) accept no liability for decisions, losses, or damages arising from use, modification, or interpretation of this material. Users remain responsible for validating any method, model, or workflow before practical application.

This notebook is a browser-friendly JupyterLite exercise based on the **Intelligent Machine Learning** part of the course. It covers:

- **Neural networks** — hidden layers connecting chromatographic input parameters to response functions.
- **Reinforcement learning and Q-learning** — an agent learns useful actions from observations and rewards.
- **Actor–critic intuition** — separating action choice from value estimation.
- **Recurrent neural networks** — memory for sequential data such as chromatograms or time series.
- **Transformers and attention** — context-dependent weighting of input tokens or chromatographic descriptors.

All examples are synthetic and lightweight. They are designed to run locally in the browser through JupyterLite.


## Learning goals

After this notebook, participants should be able to:

1. Explain what a hidden layer does in a neural-network model.
2. Distinguish supervised learning from reinforcement learning.
3. Interpret the terms **state**, **action**, **reward**, **policy**, and **value**.
4. Explain why memory is useful for sequential chromatographic signals.
5. Interpret transformer attention as a context-dependent weighting mechanism.
6. Relate these machine-learning concepts to liquid-phase separation method development.


In [ ]:
# Core stack for a light JupyterLite/Pyodide exercise.
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

import json
import math
import uuid
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display
from matplotlib.colors import ListedColormap, LinearSegmentedColormap

RANDOM_STATE = 7
rng = np.random.default_rng(RANDOM_STATE)

# IDEAS/HPLC 2026 visual palette.
IDEAS_CYAN = "#43CEFF"
IDEAS_MAGENTA = "#DB3FC8"
IDEAS_GOLD = "#FFC000"
IDEAS_GREEN = "#92D050"
IDEAS_TEXT = "#16495A"
IDEAS_GRID = "#E5F8FD"
IDEAS_PALE_CYAN = "#D9F7FF"
IDEAS_PALE_MAGENTA = "#F8D8F4"
IDEAS_PALE_GOLD = "#FFF2BF"
IDEAS_PALE_GREEN = "#E7F5D9"

CLASS_COLORS = [IDEAS_CYAN, IDEAS_MAGENTA, IDEAS_GOLD, IDEAS_GREEN]
REGION_COLORS = [IDEAS_PALE_CYAN, IDEAS_PALE_MAGENTA, IDEAS_PALE_GOLD, IDEAS_PALE_GREEN]
CLASS_CMAP = ListedColormap(CLASS_COLORS)
REGION_CMAP = ListedColormap(REGION_COLORS)
RESPONSE_CMAP = LinearSegmentedColormap.from_list(
    "ideas_response", ["#FFFFFF", IDEAS_PALE_CYAN, IDEAS_CYAN, IDEAS_MAGENTA]
)
VALUE_CMAP = LinearSegmentedColormap.from_list(
    "ideas_value", ["#FFFFFF", IDEAS_PALE_GREEN, IDEAS_GREEN, IDEAS_GOLD]
)

plt.rcParams.update({
    "axes.edgecolor": IDEAS_TEXT,
    "axes.labelcolor": IDEAS_TEXT,
    "axes.titlecolor": IDEAS_TEXT,
    "xtick.color": IDEAS_TEXT,
    "ytick.color": IDEAS_TEXT,
    "grid.color": IDEAS_GRID,
    "grid.alpha": 0.7,
    "legend.frameon": True,
})

print("Notebook style and scientific stack loaded.")


## 1. Synthetic chromatographic response function

The neural-network slide represents a typical separation problem:

```text
chromatographic optimization parameters → hidden layer(s) → sub-criteria → chromatographic response function
```

The synthetic dataset below imitates a method-development setting. The inputs are chromatographic settings. The output is a single response score combining useful resolution with penalties for long analysis time and high pressure.

The numbers are chemically plausible teaching ranges, not validated experimental data.


In [ ]:
PARAMETER_INFO = pd.DataFrame([
    ["gradient_time_min", "Gradient time", "min", "5–30"],
    ["start_B_percent", "Initial organic modifier", "% B", "5–30"],
    ["end_B_percent", "Final organic modifier", "% B", "60–95"],
    ["pH", "Mobile-phase pH", "unitless", "2–10"],
    ["temperature_C", "Column temperature", "°C", "20–60"],
    ["flow_mL_min", "Flow rate", "mL/min", "0.20–1.20"],
], columns=["code name", "meaning", "unit", "teaching range"])
PARAMETER_INFO


In [ ]:
PARAMETER_NAMES = [
    "gradient_time_min", "start_B_percent", "end_B_percent",
    "pH", "temperature_C", "flow_mL_min"
]

PARAMETER_LABELS = {
    "gradient_time_min": "Gradient time (min)",
    "start_B_percent": "Initial organic modifier (%B)",
    "end_B_percent": "Final organic modifier (%B)",
    "pH": "pH (unitless)",
    "temperature_C": "Column temperature (°C)",
    "flow_mL_min": "Flow rate (mL/min)",
}

BOUNDS = {
    "gradient_time_min": (5.0, 30.0),
    "start_B_percent": (5.0, 30.0),
    "end_B_percent": (60.0, 95.0),
    "pH": (2.0, 10.0),
    "temperature_C": (20.0, 60.0),
    "flow_mL_min": (0.20, 1.20),
}


def chromatographic_response(X):
    """Synthetic chromatographic response function scaled to approximately 0–100."""
    X = np.asarray(X, dtype=float)
    gradient = X[:, 0]
    start_b = X[:, 1]
    end_b = X[:, 2]
    ph = X[:, 3]
    temp = X[:, 4]
    flow = X[:, 5]

    # Useful separation terms: broad optimum around sensible method conditions.
    resolution = (
        55 * np.exp(-((gradient - 18) / 7.0) ** 2)
        + 22 * np.exp(-((ph - 6.4) / 1.35) ** 2)
        + 12 * np.exp(-((temp - 38) / 9.0) ** 2)
        + 8 * np.exp(-((end_b - 82) / 9.0) ** 2)
    )

    # Selectivity interaction: pH and gradient jointly matter.
    interaction = 15 * np.exp(-((gradient - 20) / 6.0) ** 2 - ((ph - 7.1) / 1.0) ** 2)

    # Penalties: very long methods, excessive pressure, and weak gradient span.
    pressure_proxy = 18 * (flow / 1.2) ** 2 * (35 / np.maximum(temp, 1))
    time_penalty = 0.80 * gradient
    weak_gradient_penalty = np.maximum(0, 40 - (end_b - start_b)) * 0.35

    response = resolution + interaction - time_penalty - pressure_proxy - weak_gradient_penalty + 20
    return np.clip(response, 0, 100)


def sample_methods(n=260, random_state=RANDOM_STATE, noise=2.5):
    local_rng = np.random.default_rng(random_state)
    X = np.column_stack([
        local_rng.uniform(*BOUNDS[name], size=n) for name in PARAMETER_NAMES
    ])
    y_clean = chromatographic_response(X)
    y = np.clip(y_clean + local_rng.normal(0, noise, size=n), 0, 100)
    return pd.DataFrame(X, columns=PARAMETER_NAMES), y

X_methods, y_response = sample_methods(n=280, noise=2.0)
method_data = X_methods.copy()
method_data["response_score"] = y_response
method_data.head()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
plot_pairs = [
    ("gradient_time_min", "pH"),
    ("flow_mL_min", "temperature_C"),
    ("start_B_percent", "end_B_percent"),
]
for ax, (xname, yname) in zip(axes, plot_pairs):
    sc = ax.scatter(
        method_data[xname], method_data[yname], c=method_data["response_score"],
        cmap=RESPONSE_CMAP, s=34, edgecolor="white", linewidth=0.5
    )
    ax.set_xlabel(PARAMETER_LABELS[xname])
    ax.set_ylabel(PARAMETER_LABELS[yname])
    ax.set_title("Response-coloured method settings")
    ax.grid(True, linewidth=0.6)
fig.colorbar(sc, ax=axes, shrink=0.85, label="Response score (0–100)")
fig.subplots_adjust(wspace=0.35, right=0.88)
plt.show()

print("Top five synthetic methods in this sample:")
display(method_data.sort_values("response_score", ascending=False).head(5).round(2))


## 2. Neural network from scratch

To keep the notebook browser-safe, the neural network below is implemented directly in NumPy. It is intentionally small:

```text
6 chromatographic inputs → 8 hidden units → 1 response prediction
```

The hidden layer is not magical. It is a set of learned intermediate features. In this example, the hidden units combine pH, gradient time, temperature, flow, and organic-modifier settings into patterns that help predict the final response score.


In [ ]:
def train_test_split_manual(X, y, test_fraction=0.25, random_state=RANDOM_STATE):
    local_rng = np.random.default_rng(random_state)
    idx = np.arange(len(y))
    local_rng.shuffle(idx)
    n_test = int(round(len(y) * test_fraction))
    test_idx = idx[:n_test]
    train_idx = idx[n_test:]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]


def standardize_train_test(X_train, X_test):
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std[std == 0] = 1
    return (X_train - mean) / std, (X_test - mean) / std, mean, std


def train_tiny_mlp(X, y, hidden=8, epochs=850, learning_rate=0.035, random_state=RANDOM_STATE):
    local_rng = np.random.default_rng(random_state)
    X_np = np.asarray(X, dtype=float)
    y_np = np.asarray(y, dtype=float).reshape(-1, 1)

    X_train, X_test, y_train, y_test = train_test_split_manual(X_np, y_np, random_state=random_state)
    X_train_s, X_test_s, X_mean, X_std = standardize_train_test(X_train, X_test)
    y_mean = y_train.mean()
    y_std = y_train.std() or 1.0
    y_train_s = (y_train - y_mean) / y_std
    y_test_s = (y_test - y_mean) / y_std

    n_features = X_train_s.shape[1]
    W1 = local_rng.normal(0, 0.35, size=(n_features, hidden))
    b1 = np.zeros((1, hidden))
    W2 = local_rng.normal(0, 0.35, size=(hidden, 1))
    b2 = np.zeros((1, 1))

    history = []
    for epoch in range(epochs):
        H = np.tanh(X_train_s @ W1 + b1)
        y_hat = H @ W2 + b2
        error = y_hat - y_train_s
        loss = float(np.mean(error ** 2))

        d_yhat = 2 * error / len(X_train_s)
        dW2 = H.T @ d_yhat
        db2 = d_yhat.sum(axis=0, keepdims=True)
        dH = d_yhat @ W2.T
        dZ1 = dH * (1 - H ** 2)
        dW1 = X_train_s.T @ dZ1
        db1 = dZ1.sum(axis=0, keepdims=True)

        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1
        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2

        if epoch % 25 == 0 or epoch == epochs - 1:
            H_test = np.tanh(X_test_s @ W1 + b1)
            pred_test_s = H_test @ W2 + b2
            test_loss = float(np.mean((pred_test_s - y_test_s) ** 2))
            history.append((epoch, loss, test_loss))

    model = {
        "W1": W1, "b1": b1, "W2": W2, "b2": b2,
        "X_mean": X_mean, "X_std": X_std, "y_mean": y_mean, "y_std": y_std,
        "history": pd.DataFrame(history, columns=["epoch", "train_mse", "test_mse"]),
        "X_test": X_test, "y_test": y_test,
    }
    return model


def mlp_predict(model, X):
    X = np.asarray(X, dtype=float)
    Xs = (X - model["X_mean"]) / model["X_std"]
    H = np.tanh(Xs @ model["W1"] + model["b1"])
    y_pred = H @ model["W2"] + model["b2"]
    return (y_pred.ravel() * model["y_std"] + model["y_mean"]), H

mlp_model = train_tiny_mlp(X_methods.values, y_response, hidden=8, epochs=850, learning_rate=0.035)

history = mlp_model["history"]
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(history["epoch"], history["train_mse"], color=IDEAS_CYAN, linewidth=2, label="Training loss")
ax.plot(history["epoch"], history["test_mse"], color=IDEAS_MAGENTA, linewidth=2, label="Test loss")
ax.set_xlabel("Training epoch")
ax.set_ylabel("Mean-squared error on standardized response")
ax.set_title("Tiny neural network learning curve")
ax.grid(True, linewidth=0.6)
ax.legend()
plt.tight_layout()
plt.show()

pred_test, hidden_test = mlp_predict(mlp_model, mlp_model["X_test"])
true_test = mlp_model["y_test"].ravel()
rmse = np.sqrt(np.mean((pred_test - true_test) ** 2))
print(f"Test RMSE: {rmse:.2f} response-score units")


In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 5.2))
ax.scatter(true_test, pred_test, color=IDEAS_CYAN, edgecolor="white", linewidth=0.6, s=38, alpha=0.85)
ax.plot([0, 100], [0, 100], color=IDEAS_MAGENTA, linewidth=2, label="Perfect prediction")
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)
ax.set_xlabel("True response score")
ax.set_ylabel("Predicted response score")
ax.set_title("Neural-network response prediction")
ax.grid(True, linewidth=0.6)
ax.legend()
plt.tight_layout()
plt.show()

# Hidden-layer interpretation: compare a poor and strong method.
order = np.argsort(true_test)
example_idx = [order[0], order[-1]]
labels = ["Low-response method", "High-response method"]
activations = hidden_test[example_idx]

fig, ax = plt.subplots(figsize=(8, 3.6))
width = 0.35
xpos = np.arange(activations.shape[1])
ax.bar(xpos - width/2, activations[0], width=width, color=IDEAS_GOLD, label=labels[0])
ax.bar(xpos + width/2, activations[1], width=width, color=IDEAS_GREEN, label=labels[1])
ax.axhline(0, color=IDEAS_TEXT, linewidth=0.8)
ax.set_xticks(xpos)
ax.set_xticklabels([f"H{i+1}" for i in xpos])
ax.set_ylabel("Hidden-unit activation")
ax.set_title("The hidden layer creates intermediate features")
ax.grid(True, axis="y", linewidth=0.6)
ax.legend()
plt.tight_layout()
plt.show()

print("Interpretation: hidden units are learned intermediate signals. They are not directly pH, flow, or gradient time, but combinations that help predict response.")


## 3. Reinforcement learning and Q-learning

In reinforcement learning, the model does not only fit labelled examples. It acts, observes the consequence, receives a reward, and updates future behaviour.

For a chromatographic analogy, imagine an agent adjusting a method over a small experimental map. The state is the current method setting. The actions move to neighbouring method settings. The reward is the chromatographic response score obtained after the action.


In [ ]:
# A small method-development environment.
GRADIENT_GRID = np.linspace(8, 28, 9)     # min
PH_GRID = np.linspace(3.0, 9.0, 9)        # unitless
ACTIONS = ["up", "right", "down", "left"]
ACTION_DELTAS = {
    "up": (-1, 0),       # lower pH-index row in the display
    "right": (0, 1),
    "down": (1, 0),
    "left": (0, -1),
}


def state_to_method(row, col):
    # Fixed values for variables not shown in the 2D map.
    pH = PH_GRID[row]
    gradient = GRADIENT_GRID[col]
    return np.array([[gradient, 12, 84, pH, 38, 0.65]])


def reward_at_state(row, col):
    # Scale response to a reinforcement-learning reward range.
    raw = chromatographic_response(state_to_method(row, col))[0]
    return (raw - 45.0) / 20.0

REWARD_MAP = np.array([[reward_at_state(r, c) for c in range(len(GRADIENT_GRID))] for r in range(len(PH_GRID))])


def step(state, action_index):
    row, col = state
    dr, dc = ACTION_DELTAS[ACTIONS[action_index]]
    new_row = int(np.clip(row + dr, 0, len(PH_GRID) - 1))
    new_col = int(np.clip(col + dc, 0, len(GRADIENT_GRID) - 1))
    reward = reward_at_state(new_row, new_col)
    return (new_row, new_col), reward


def train_q_learning(episodes=450, alpha=0.30, gamma=0.88, epsilon=0.25, random_state=RANDOM_STATE):
    local_rng = np.random.default_rng(random_state)
    n_rows, n_cols = REWARD_MAP.shape
    Q = np.zeros((n_rows, n_cols, len(ACTIONS)))
    returns = []

    for ep in range(episodes):
        state = (local_rng.integers(0, n_rows), local_rng.integers(0, n_cols))
        total_reward = 0.0
        for _ in range(18):
            r, c = state
            if local_rng.random() < epsilon:
                action_index = local_rng.integers(len(ACTIONS))
            else:
                action_index = int(np.argmax(Q[r, c]))
            new_state, reward = step(state, action_index)
            nr, nc = new_state
            Q[r, c, action_index] += alpha * (reward + gamma * np.max(Q[nr, nc]) - Q[r, c, action_index])
            state = new_state
            total_reward += reward
        returns.append(total_reward)
    return Q, np.array(returns)

Q, q_returns = train_q_learning()
POLICY = np.argmax(Q, axis=2)
VALUE = np.max(Q, axis=2)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

im0 = axes[0].imshow(REWARD_MAP, origin="lower", cmap=RESPONSE_CMAP)
axes[0].set_title("Environment reward map")
axes[0].set_xlabel("Gradient time (min)")
axes[0].set_ylabel("pH")
axes[0].set_xticks(range(len(GRADIENT_GRID)))
axes[0].set_xticklabels([f"{v:.0f}" for v in GRADIENT_GRID])
axes[0].set_yticks(range(len(PH_GRID)))
axes[0].set_yticklabels([f"{v:.1f}" for v in PH_GRID])
fig.colorbar(im0, ax=axes[0], shrink=0.8, label="Reward")

axes[1].plot(q_returns, color=IDEAS_MAGENTA, linewidth=1.5)
axes[1].plot(pd.Series(q_returns).rolling(30, min_periods=1).mean(), color=IDEAS_CYAN, linewidth=2.5, label="30-episode average")
axes[1].set_title("Q-learning reward during training")
axes[1].set_xlabel("Episode")
axes[1].set_ylabel("Total reward per episode")
axes[1].grid(True, linewidth=0.6)
axes[1].legend()

fig.subplots_adjust(wspace=0.35, right=0.90)
plt.show()


In [ ]:
def plot_policy(ax, policy, title="Learned Q-learning policy"):
    im = ax.imshow(REWARD_MAP, origin="lower", cmap=RESPONSE_CMAP, alpha=0.75)
    arrow = {"up": "↑", "right": "→", "down": "↓", "left": "←"}
    for r in range(policy.shape[0]):
        for c in range(policy.shape[1]):
            action = ACTIONS[int(policy[r, c])]
            ax.text(c, r, arrow[action], ha="center", va="center", fontsize=16, color=IDEAS_TEXT, fontweight="bold")
    ax.set_title(title)
    ax.set_xlabel("Gradient time (min)")
    ax.set_ylabel("pH")
    ax.set_xticks(range(len(GRADIENT_GRID)))
    ax.set_xticklabels([f"{v:.0f}" for v in GRADIENT_GRID])
    ax.set_yticks(range(len(PH_GRID)))
    ax.set_yticklabels([f"{v:.1f}" for v in PH_GRID])
    return im

fig, ax = plt.subplots(figsize=(6.4, 5.2))
plot_policy(ax, POLICY)
plt.tight_layout()
plt.show()

best_state = np.unravel_index(np.argmax(REWARD_MAP), REWARD_MAP.shape)
print(f"Best synthetic grid point: pH={PH_GRID[best_state[0]]:.1f}, gradient time={GRADIENT_GRID[best_state[1]]:.0f} min")
print("Interpretation: arrows form a policy. They are not fitted labels; they are preferred actions learned from reward feedback.")


## 4. Actor–critic and actual PPO training

The PPO slide separates the reinforcement-learning system into two roles:

- **Actor:** a policy that selects the next action.
- **Critic:** a value function that estimates how promising the current state is.

Here we train a small **tabular PPO-style actor–critic** model in pure NumPy. The environment is the same simplified chromatographic method-development grid used above: the agent moves through pH and gradient-time settings and receives a reward from the synthetic chromatographic response.

This is deliberately small and browser-safe. It is not meant to be production reinforcement learning; it is a transparent teaching implementation of the PPO idea: update the actor, but restrict the update with a clipped probability ratio so the policy does not change too abruptly.


In [ ]:
def softmax_vector(x):
    z = x - np.max(x)
    exp_z = np.exp(z)
    return exp_z / np.sum(exp_z)


def flatten_state(state):
    row, col = state
    return row * len(GRADIENT_GRID) + col


def unflatten_state(index):
    row = index // len(GRADIENT_GRID)
    col = index % len(GRADIENT_GRID)
    return int(row), int(col)


def discounted_returns(rewards, gamma=0.90):
    out = np.zeros(len(rewards), dtype=float)
    running = 0.0
    for i in range(len(rewards) - 1, -1, -1):
        running = rewards[i] + gamma * running
        out[i] = running
    return out


def train_tabular_ppo(
    iterations=90,
    rollouts_per_iteration=10,
    horizon=16,
    gamma=0.90,
    clip_epsilon=0.18,
    actor_lr=0.035,
    critic_lr=0.080,
    update_epochs=3,
    random_state=RANDOM_STATE,
):
    """Train a compact PPO-style actor–critic model on the method grid.

    The actor is a table of action logits for each state. The critic is a table
    of scalar values for each state. This makes the algorithm visible and keeps
    the exercise suitable for JupyterLite.
    """
    local_rng = np.random.default_rng(random_state)
    n_rows, n_cols = REWARD_MAP.shape
    n_states = n_rows * n_cols
    n_actions = len(ACTIONS)

    actor_logits = np.zeros((n_states, n_actions), dtype=float)
    critic_values = np.zeros(n_states, dtype=float)
    mean_returns = []

    for _iteration in range(iterations):
        batch_states = []
        batch_actions = []
        batch_old_probs = []
        batch_returns = []
        episode_totals = []

        # Collect rollouts using the current actor.
        for _rollout in range(rollouts_per_iteration):
            state = (local_rng.integers(0, n_rows), local_rng.integers(0, n_cols))
            states = []
            actions = []
            old_probs = []
            rewards = []

            for _step in range(horizon):
                state_index = flatten_state(state)
                probs = softmax_vector(actor_logits[state_index])
                action_index = int(local_rng.choice(n_actions, p=probs))
                next_state, reward = step(state, action_index)

                states.append(state_index)
                actions.append(action_index)
                old_probs.append(probs[action_index])
                rewards.append(reward)

                state = next_state

            returns = discounted_returns(rewards, gamma=gamma)
            batch_states.extend(states)
            batch_actions.extend(actions)
            batch_old_probs.extend(old_probs)
            batch_returns.extend(returns)
            episode_totals.append(np.sum(rewards))

        batch_states = np.asarray(batch_states, dtype=int)
        batch_actions = np.asarray(batch_actions, dtype=int)
        batch_old_probs = np.asarray(batch_old_probs, dtype=float)
        batch_returns = np.asarray(batch_returns, dtype=float)

        advantages = batch_returns - critic_values[batch_states]
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-9)

        # PPO-style clipped actor update and critic update.
        for _epoch in range(update_epochs):
            order = local_rng.permutation(len(batch_states))
            for i in order:
                s = batch_states[i]
                a = batch_actions[i]
                old_prob = max(batch_old_probs[i], 1e-9)
                adv = advantages[i]

                probs = softmax_vector(actor_logits[s])
                ratio = probs[a] / old_prob

                # Clipping: when the ratio has already moved too far in the
                # advantage-improving direction, suppress that actor gradient.
                use_actor_gradient = True
                if adv > 0 and ratio > 1.0 + clip_epsilon:
                    use_actor_gradient = False
                if adv < 0 and ratio < 1.0 - clip_epsilon:
                    use_actor_gradient = False

                if use_actor_gradient:
                    grad_log_prob = -probs
                    grad_log_prob[a] += 1.0
                    actor_logits[s] += actor_lr * ratio * adv * grad_log_prob

                value_error = batch_returns[i] - critic_values[s]
                critic_values[s] += critic_lr * value_error

        mean_returns.append(float(np.mean(episode_totals)))

    return actor_logits, critic_values, np.array(mean_returns)


PPO_LOGITS, PPO_VALUES_FLAT, ppo_returns = train_tabular_ppo()
PPO_POLICY = np.argmax(PPO_LOGITS, axis=1).reshape(REWARD_MAP.shape)
PPO_VALUE = PPO_VALUES_FLAT.reshape(REWARD_MAP.shape)

selected_state = (5, 4)  # pH index, gradient-time index
selected_index = flatten_state(selected_state)
selected_action_probs = softmax_vector(PPO_LOGITS[selected_index])

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

axes[0].plot(ppo_returns, color=IDEAS_MAGENTA, linewidth=1.4, label="Episode-batch mean")
axes[0].plot(pd.Series(ppo_returns).rolling(10, min_periods=1).mean(), color=IDEAS_CYAN, linewidth=2.4, label="10-iteration average")
axes[0].set_title("PPO training return")
axes[0].set_xlabel("PPO iteration")
axes[0].set_ylabel("Mean rollout reward")
axes[0].grid(True, linewidth=0.6)
axes[0].legend(fontsize=8)

plot_policy(axes[1], PPO_POLICY, title="PPO actor: learned policy")

im = axes[2].imshow(PPO_VALUE, origin="lower", cmap=VALUE_CMAP)
axes[2].scatter(selected_state[1], selected_state[0], marker="o", s=150, color=IDEAS_MAGENTA, edgecolor="white", linewidth=1.2)
axes[2].set_title("PPO critic: learned value map")
axes[2].set_xlabel("Gradient time (min)")
axes[2].set_ylabel("pH")
axes[2].set_xticks(range(len(GRADIENT_GRID)))
axes[2].set_xticklabels([f"{v:.0f}" for v in GRADIENT_GRID])
axes[2].set_yticks(range(len(PH_GRID)))
axes[2].set_yticklabels([f"{v:.1f}" for v in PH_GRID])
fig.colorbar(im, ax=axes[2], shrink=0.80, label="Critic value")

fig.subplots_adjust(wspace=0.38, right=0.92)
plt.show()

plt.figure(figsize=(6.4, 4.1))
plt.bar(ACTIONS, selected_action_probs, color=CLASS_COLORS)
plt.ylim(0, 1)
plt.ylabel("Action probability")
plt.title(f"PPO actor probabilities at pH={PH_GRID[selected_state[0]]:.1f}, gradient={GRADIENT_GRID[selected_state[1]]:.0f} min")
plt.grid(True, axis="y", linewidth=0.6)
plt.tight_layout()
plt.show()

best_state = np.unravel_index(np.argmax(REWARD_MAP), REWARD_MAP.shape)
print(f"Best synthetic grid point: pH={PH_GRID[best_state[0]]:.1f}, gradient time={GRADIENT_GRID[best_state[1]]:.0f} min")
print(f"Final PPO mean rollout reward: {ppo_returns[-1]:.2f}")
print("Interpretation: the PPO actor learns an action policy, while the PPO critic learns a value map. The clipping rule limits overly large policy updates.")


## 5. Recurrent neural-network baseline training from blank chromatograms

RNNs are designed for sequential data. In liquid-phase separations, examples include detector traces, gradient programs, injection sequences, and time-dependent instrument states.

This section **trains a small recurrent neural network from scratch in NumPy**. The teaching task is baseline estimation.

Important chromatographic assumption: the RNN is trained on **blank chromatograms only**. These blank runs contain baseline drift and detector noise, but **no analyte peaks**. After training, the model is applied to chromatograms that do contain peaks. This keeps the teaching logic realistic: the model learns the background from blanks rather than learning analyte peaks as if they were part of the baseline.

This is not a production baseline-correction algorithm. It is a compact demonstration of how an RNN learns from many blank sequences by adjusting weights through backpropagation through time.


In [ ]:
def gaussian(x, center, width, height):
    return height * np.exp(-0.5 * ((x - center) / width) ** 2)


def make_chromatogram_sequence(drift_strength=0.55, noise=0.018, random_state=RANDOM_STATE):
    """Single sample chromatogram used by the static plots and the interactive exercise."""
    local_rng = np.random.default_rng(random_state)
    time = np.linspace(0, 12, 360)
    baseline = 0.12 + drift_strength * (time / time.max()) ** 1.6
    signal = baseline.copy()
    signal += gaussian(time, 2.4, 0.18, 0.95)
    signal += gaussian(time, 4.1, 0.28, 0.55)
    signal += gaussian(time, 7.6, 0.22, 1.10)
    signal += gaussian(time, 9.2, 0.35, 0.65)
    signal += local_rng.normal(0, noise, size=time.size)
    return time, signal, baseline



def make_rnn_training_dataset(
    n_sequences=90,
    n_points=96,
    noise=0.020,
    include_peaks=False,
    random_state=RANDOM_STATE,
):
    """Generate synthetic chromatographic sequences with known baselines.

    By default this function generates **blank chromatograms**: baseline drift plus
    detector noise, without analyte peaks. That is the correct training set for the
    RNN baseline model in this course example.

    Set include_peaks=True only for application/evaluation sequences that mimic real
    sample chromatograms. Those sequences are deliberately not used for training.
    """
    local_rng = np.random.default_rng(random_state)
    time = np.linspace(0, 12, n_points)
    signals = []
    baselines = []

    for _ in range(n_sequences):
        drift_strength = local_rng.uniform(0.22, 1.05)
        curvature = local_rng.uniform(1.10, 2.00)
        baseline = 0.08 + drift_strength * (time / time.max()) ** curvature
        baseline += local_rng.uniform(-0.035, 0.035) * np.sin(
            2 * np.pi * time / time.max() + local_rng.uniform(0, 2 * np.pi)
        )

        signal = baseline.copy()
        if include_peaks:
            for _peak in range(local_rng.integers(3, 6)):
                signal += gaussian(
                    time,
                    center=local_rng.uniform(1.2, 10.8),
                    width=local_rng.uniform(0.09, 0.36),
                    height=local_rng.uniform(0.18, 1.05),
                )
        signal += local_rng.normal(0, noise, size=n_points)

        signals.append(signal)
        baselines.append(baseline)

    return time, np.array(signals), np.array(baselines)


def rnn_forward_batch(X_sequence, params):
    """Forward pass for a simple Elman RNN over a full batch of sequences."""
    Wxh, Whh, bh, Why, by = (
        params["Wxh"],
        params["Whh"],
        params["bh"],
        params["Why"],
        params["by"],
    )
    n_sequences, n_time = X_sequence.shape
    hidden_size = len(bh)
    hidden = np.zeros((n_time + 1, n_sequences, hidden_size))
    output = np.zeros((n_sequences, n_time))

    for t in range(n_time):
        hidden[t + 1] = np.tanh(
            X_sequence[:, t, None] * Wxh[:, 0]
            + hidden[t] @ Whh.T
            + bh
        )
        output[:, t] = hidden[t + 1] @ Why[0] + by[0]

    return output, hidden


def train_rnn_baseline_model(
    X_train,
    y_train,
    hidden_size=8,
    epochs=120,
    learning_rate=0.012,
    random_state=RANDOM_STATE,
):
    """Train a small recurrent model by backpropagation through time."""
    local_rng = np.random.default_rng(random_state)
    params = {
        "Wxh": local_rng.normal(0, 0.20, size=(hidden_size, 1)),
        "Whh": local_rng.normal(0, 0.05, size=(hidden_size, hidden_size)),
        "bh": np.zeros(hidden_size),
        "Why": local_rng.normal(0, 0.20, size=(1, hidden_size)),
        "by": np.zeros(1),
    }

    # Adam optimizer state. This keeps the example short while making training stable.
    m = {key: np.zeros_like(value) for key, value in params.items()}
    v = {key: np.zeros_like(value) for key, value in params.items()}
    beta1, beta2, eps = 0.9, 0.999, 1e-8
    losses = []
    n_sequences, n_time = X_train.shape

    for epoch in range(1, epochs + 1):
        y_pred, hidden = rnn_forward_batch(X_train, params)
        residual = y_pred - y_train
        losses.append(float(np.mean(residual ** 2)))

        # d/dy mean((y_pred - y_true)^2)
        doutput = 2 * residual / (n_sequences * n_time)
        grads = {key: np.zeros_like(value) for key, value in params.items()}
        dh_next = np.zeros((n_sequences, hidden_size))

        # Backpropagation through time.
        for t in range(n_time - 1, -1, -1):
            grads["Why"][0] += doutput[:, t] @ hidden[t + 1]
            grads["by"][0] += doutput[:, t].sum()

            dh = doutput[:, t, None] * params["Why"][0] + dh_next
            dh_raw = (1 - hidden[t + 1] ** 2) * dh

            grads["Wxh"][:, 0] += np.sum(dh_raw * X_train[:, t, None], axis=0)
            grads["Whh"] += dh_raw.T @ hidden[t]
            grads["bh"] += dh_raw.sum(axis=0)
            dh_next = dh_raw @ params["Whh"]

        # Gradient clipping and Adam update.
        for key in params:
            np.clip(grads[key], -1.0, 1.0, out=grads[key])
            m[key] = beta1 * m[key] + (1 - beta1) * grads[key]
            v[key] = beta2 * v[key] + (1 - beta2) * (grads[key] ** 2)
            m_hat = m[key] / (1 - beta1 ** epoch)
            v_hat = v[key] / (1 - beta2 ** epoch)
            params[key] -= learning_rate * m_hat / (np.sqrt(v_hat) + eps)

    return params, np.array(losses)


# Build a small sequence dataset from blank chromatograms only: blank signal -> true baseline.
time_rnn, blank_sequences, blank_baseline_targets = make_rnn_training_dataset(
    n_sequences=90,
    n_points=96,
    include_peaks=False,
    random_state=RANDOM_STATE,
)

n_train = 70
X_train_raw, X_blank_test_raw = blank_sequences[:n_train], blank_sequences[n_train:]
y_train_raw, y_blank_test_raw = blank_baseline_targets[:n_train], blank_baseline_targets[n_train:]

# Separate application set: sample chromatograms with peaks. These are not used for training.
_, X_sample_raw, y_sample_raw = make_rnn_training_dataset(
    n_sequences=16,
    n_points=96,
    include_peaks=True,
    random_state=RANDOM_STATE + 31,
)

# Normalize using the blank training set only.
x_mean, x_std = X_train_raw.mean(), X_train_raw.std()
y_mean, y_std = y_train_raw.mean(), y_train_raw.std()
X_train = (X_train_raw - x_mean) / x_std
X_blank_test = (X_blank_test_raw - x_mean) / x_std
X_sample = (X_sample_raw - x_mean) / x_std
y_train = (y_train_raw - y_mean) / y_std
y_blank_test = (y_blank_test_raw - y_mean) / y_std

params_rnn, rnn_losses = train_rnn_baseline_model(
    X_train,
    y_train,
    hidden_size=8,
    epochs=120,
    learning_rate=0.012,
    random_state=RANDOM_STATE,
)

train_prediction_norm, _ = rnn_forward_batch(X_train, params_rnn)
blank_test_prediction_norm, _ = rnn_forward_batch(X_blank_test, params_rnn)
sample_prediction_norm, sample_hidden = rnn_forward_batch(X_sample, params_rnn)
train_prediction = train_prediction_norm * y_std + y_mean
blank_test_prediction = blank_test_prediction_norm * y_std + y_mean
sample_prediction = sample_prediction_norm * y_std + y_mean

train_rmse = np.sqrt(np.mean((train_prediction - y_train_raw) ** 2))
blank_test_rmse = np.sqrt(np.mean((blank_test_prediction - y_blank_test_raw) ** 2))
sample_rmse = np.sqrt(np.mean((sample_prediction - y_sample_raw) ** 2))
example_index = 3

print(f"Blank training sequences: {n_train}")
print(f"Blank test sequences: {len(X_blank_test_raw)}")
print(f"Peaked sample sequences used only after training: {len(X_sample_raw)}")
print(f"RNN hidden units: 8")
print(f"Final normalized training loss: {rnn_losses[-1]:.4f}")
print(f"Train blank-baseline RMSE: {train_rmse:.3f} detector-response units")
print(f"Blank-test baseline RMSE: {blank_test_rmse:.3f} detector-response units")
print(f"Peaked-sample baseline RMSE: {sample_rmse:.3f} detector-response units")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
axes[0].plot(rnn_losses, color=IDEAS_MAGENTA, linewidth=2)
axes[0].set_xlabel("Training epoch")
axes[0].set_ylabel("Mean squared error (normalized)")
axes[0].set_title("RNN training curve: blank chromatograms only")
axes[0].grid(True, linewidth=0.6)

axes[1].plot(
    time_rnn,
    X_sample_raw[example_index],
    color=IDEAS_CYAN,
    linewidth=1.4,
    label="Observed sample chromatogram",
)
axes[1].plot(
    time_rnn,
    y_sample_raw[example_index],
    color=IDEAS_GOLD,
    linewidth=2.2,
    label="True synthetic baseline",
)
axes[1].plot(
    time_rnn,
    sample_prediction[example_index],
    color=IDEAS_MAGENTA,
    linewidth=2.0,
    linestyle="--",
    label="RNN baseline prediction",
)
axes[1].set_xlabel("Time (min)")
axes[1].set_ylabel("Detector response (a.u.)")
axes[1].set_title("Blank-trained RNN applied to a chromatogram with peaks")
axes[1].grid(True, linewidth=0.6)
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

# Interpretation of the trained RNN example.
print(
    "Interpretation: the RNN is trained on blank chromatograms only, so the training target "
    "contains baseline drift and detector noise but no analyte peaks. Applying the trained "
    "model to sample chromatograms with peaks illustrates the chromatographic idea: learn "
    "the background from blanks, then use that learned background model when interpreting samples."
)


## 6. Transformer-style attention

Transformers replace recurrence with self-attention. The key idea is not that words or descriptors are processed only one by one. Instead, each token can weight other tokens according to context.

For a chromatographic analogy, a token such as **resolution** should attend strongly to parameters that influence selectivity and separation quality, while **pressure** should attend strongly to flow and temperature.


In [ ]:
TOKENS = ["gradient", "pH", "temperature", "flow", "pressure", "resolution", "analysis time"]

# Hand-crafted token embeddings for teaching. Dimensions are not physical units;
# they encode synthetic concepts such as selectivity, speed, pressure, and thermal effect.
EMBEDDINGS = np.array([
    [0.95, 0.30, 0.20, 0.55],  # gradient
    [0.80, 0.95, 0.10, 0.15],  # pH
    [0.35, 0.35, 0.75, 0.40],  # temperature
    [0.10, 0.10, 0.95, 0.90],  # flow
    [0.05, 0.10, 1.00, 0.65],  # pressure
    [1.00, 0.80, 0.20, 0.15],  # resolution
    [0.40, 0.05, 0.20, 1.00],  # analysis time
])


def attention_matrix(embeddings, focus_boost=1.0):
    Q = embeddings.copy()
    K = embeddings.copy()
    scores = (Q @ K.T) / np.sqrt(embeddings.shape[1])
    # Make the teaching example slightly sharper without changing the basic formula.
    scores *= focus_boost
    scores = scores - scores.max(axis=1, keepdims=True)
    weights = np.exp(scores)
    weights = weights / weights.sum(axis=1, keepdims=True)
    return weights

ATTN = attention_matrix(EMBEDDINGS, focus_boost=3.0)

fig, ax = plt.subplots(figsize=(7.5, 6.2))
im = ax.imshow(ATTN, cmap=RESPONSE_CMAP, vmin=0, vmax=ATTN.max())
ax.set_xticks(range(len(TOKENS)))
ax.set_xticklabels(TOKENS, rotation=35, ha="right")
ax.set_yticks(range(len(TOKENS)))
ax.set_yticklabels(TOKENS)
ax.set_title("Transformer-style self-attention weights")
for i in range(len(TOKENS)):
    for j in range(len(TOKENS)):
        ax.text(j, i, f"{ATTN[i, j]*100:.0f}%", ha="center", va="center", fontsize=8, color=IDEAS_TEXT)
fig.colorbar(im, ax=ax, shrink=0.8, label="Attention weight")
fig.subplots_adjust(left=0.18, bottom=0.24, right=0.86, top=0.90)
plt.show()

resolution_row = pd.Series(ATTN[TOKENS.index("resolution")], index=TOKENS).sort_values(ascending=False)
print("Highest attention weights for the token 'resolution':")
display((100 * resolution_row).round(1).to_frame("attention (%)"))


## 7. Interactive intelligent-machine-learning exercise

Run the cell below to open a course exercise dashboard. Students can switch between neural networks, Q-learning, recurrent neural networks, and transformer attention without editing code.

Each module has its own dynamic controls. The **RNN panel includes training with different blank-training-set sizes, recurrent hidden-unit counts, and training epochs**. The RNN training data are **blank chromatograms only**: baseline drift and detector noise without analyte peaks. The trained model is then shown on sample chromatograms that contain peaks. Section 5 above contains the full NumPy Elman-RNN training example.


In [ ]:
# Interactive intelligent-machine-learning exercise with dynamic controls for each module, including blank-trained RNN variants.
# Run this cell once. Students do not need to edit the code below.

print("Preparing dynamic interactive exercise. This may take a few seconds in JupyterLite...")

# -----------------------------------------------------------------------------
# 1. Neural-network payload: vary training-set size and network architecture.
# -----------------------------------------------------------------------------
# The earlier notebook section trains a real small neural network in NumPy. For this
# interactive exercise, the neural-network panel uses a lightweight teaching proxy so
# that the dashboard remains responsive in JupyterLite. The proxy preserves the
# key concepts: sparse data, model capacity, underfitting, and overfitting.

def make_nn_dashboard_payload():
    arch_options = {
        "1x4": {"label": "1 hidden layer, 4 units", "capacity": 0.45, "parameters": 33},
        "1x8": {"label": "1 hidden layer, 8 units", "capacity": 0.72, "parameters": 65},
        "1x16": {"label": "1 hidden layer, 16 units", "capacity": 0.93, "parameters": 129},
        "2x8_4": {"label": "2 hidden layers, 8 + 4 units", "capacity": 0.98, "parameters": 97},
    }
    training_sizes = [60, 140, 280]
    gradient_vals = np.linspace(8, 28, 30)
    ph_vals = np.linspace(3.0, 9.0, 22)

    methods = []
    for ph in ph_vals:
        for grad in gradient_vals:
            methods.append([grad, 12, 84, ph, 38, 0.65])
    methods = np.array(methods)
    actual = chromatographic_response(methods).reshape(len(ph_vals), len(gradient_vals))

    # Structured patterns used to mimic model bias and overfitting artefacts.
    G, P = np.meshgrid(gradient_vals, ph_vals)
    broad_bias = (
        10 * np.exp(-((G - 18) / 9.5) ** 2 - ((P - 6.4) / 2.4) ** 2)
        - 4 * (G - gradient_vals.mean()) / (gradient_vals.max() - gradient_vals.min())
    )
    fine_artefact = 5.0 * np.sin(1.7 * G + 0.9 * P) + 3.0 * np.cos(0.8 * G - 1.5 * P)
    local_artefact = 7.0 * np.exp(-((G - 25) / 2.2) ** 2 - ((P - 4.1) / 0.7) ** 2)

    results = {}
    for arch_key, arch in arch_options.items():
        for n_train in training_sizes:
            data_factor = np.sqrt(n_train / max(training_sizes))
            capacity = arch["capacity"]
            underfit = max(0.0, 1.05 - 0.55 * capacity - 0.55 * data_factor)
            overfit = max(0.0, capacity - data_factor) * 0.75
            predicted = actual - underfit * broad_bias + overfit * (fine_artefact + local_artefact)
            predicted = np.clip(predicted, 0, 100)
            error = predicted - actual

            # Training error can look deceptively good for high-capacity models with few data,
            # while independent-test error remains high. This is the intended teaching point.
            test_rmse = float(np.sqrt(np.mean(error ** 2)))
            train_rmse = max(1.2, test_rmse * (0.52 + 0.35 * data_factor - 0.25 * overfit))

            results[f"{arch_key}|{n_train}"] = {
                "gradient": gradient_vals.round(2).tolist(),
                "pH": ph_vals.round(2).tolist(),
                "actual": actual.round(2).tolist(),
                "predicted": predicted.round(2).tolist(),
                "error": error.round(2).tolist(),
                "train_rmse": round(float(train_rmse), 2),
                "test_rmse": round(float(test_rmse), 2),
                "n_parameters": int(arch["parameters"]),
                "architecture_label": arch["label"],
                "training_size": int(n_train),
                "underfit_index": round(float(underfit), 2),
                "overfit_index": round(float(overfit), 2),
            }

    return {
        "architectures": [{"key": k, "label": v["label"]} for k, v in arch_options.items()],
        "training_sizes": training_sizes,
        "results": results,
    }


# -----------------------------------------------------------------------------
# 2. Q-learning payload: vary training episodes and exploration rate.
# -----------------------------------------------------------------------------

def make_q_dashboard_payload():
    episode_options = [40, 150, 450]
    epsilon_options = [0.05, 0.25, 0.45]
    results = {}
    for episodes in episode_options:
        for epsilon in epsilon_options:
            Q_local, returns = train_q_learning(
                episodes=episodes,
                alpha=0.30,
                gamma=0.88,
                epsilon=epsilon,
                random_state=RANDOM_STATE + episodes + int(epsilon * 100),
            )
            policy = np.argmax(Q_local, axis=2)
            value = np.max(Q_local, axis=2)
            rolling = pd.Series(returns).rolling(20, min_periods=1).mean().to_numpy()
            results[f"{episodes}|{epsilon:.2f}"] = {
                "policy": policy.astype(int).tolist(),
                "value": value.round(3).tolist(),
                "returns": returns.round(3).tolist(),
                "rolling": rolling.round(3).tolist(),
                "last20": round(float(np.mean(returns[-min(20, len(returns)):])) if len(returns) else 0.0, 3),
                "max_value": round(float(np.max(value)), 3),
            }
    return {
        "gradient": GRADIENT_GRID.round(2).tolist(),
        "pH": PH_GRID.round(2).tolist(),
        "reward": REWARD_MAP.round(3).tolist(),
        "actions": ACTIONS,
        "episode_options": episode_options,
        "epsilon_options": epsilon_options,
        "results": results,
    }


# -----------------------------------------------------------------------------
# 3. RNN-training payload: vary blank-training-set size, recurrent hidden units, and epochs.
# -----------------------------------------------------------------------------

def make_rnn_dashboard_payload():
    """Train lightweight blank-trained recurrent models for the interactive dashboard.

    Section 5 above trains a full NumPy Elman RNN with backpropagation through time.
    The dashboard below uses a browser-accelerated recurrent architecture: several
    recurrent hidden-unit traces are generated from the chromatogram, and a linear
    output layer is trained by gradient descent to predict the baseline. Training is
    performed on blank chromatograms only, then the trained model is displayed on
    peaked sample chromatograms.
    """
    training_size_options = [12, 30, 50]
    hidden_size_options = [1, 3, 6]  # displayed as recurrent hidden units
    epoch_options = [5, 30, 80]
    max_epochs = max(epoch_options)

    time, blank_signals, blank_baselines = make_rnn_training_dataset(
        n_sequences=66,
        n_points=80,
        noise=0.024,
        include_peaks=False,
        random_state=RANDOM_STATE + 271,
    )
    _, sample_signals, sample_baselines = make_rnn_training_dataset(
        n_sequences=16,
        n_points=80,
        noise=0.024,
        include_peaks=True,
        random_state=RANDOM_STATE + 401,
    )
    test_start = 50
    X_blank_test_raw = blank_signals[test_start:]
    y_blank_test_raw = blank_baselines[test_start:]
    X_sample_raw = sample_signals
    y_sample_raw = sample_baselines
    example_index = 3

    def recurrent_feature_bank(X_norm, n_units):
        """Recurrent hidden-state generator: each unit has a different temporal scale."""
        alphas = np.linspace(0.50, 0.97, n_units)
        n_seq, n_time = X_norm.shape
        H = np.zeros((n_seq, n_time, n_units))
        for j, alpha in enumerate(alphas):
            prev = np.zeros(n_seq)
            input_fraction = 1.0 - alpha
            for t in range(n_time):
                prev = alpha * prev + input_fraction * X_norm[:, t]
                H[:, t, j] = prev
        return H

    def build_features(X_norm, n_units):
        H = recurrent_feature_bank(X_norm, n_units)
        bias = np.ones((*X_norm.shape, 1))
        return np.concatenate([H, X_norm[:, :, None], bias], axis=2)

    results = {}
    for n_train in training_size_options:
        X_train_raw = blank_signals[:n_train]
        y_train_raw = blank_baselines[:n_train]

        x_mean = float(X_train_raw.mean())
        x_std = float(X_train_raw.std() + 1e-12)
        y_mean = float(y_train_raw.mean())
        y_std = float(y_train_raw.std() + 1e-12)
        X_train = (X_train_raw - x_mean) / x_std
        y_train = (y_train_raw - y_mean) / y_std
        X_blank_test = (X_blank_test_raw - x_mean) / x_std
        X_sample = (X_sample_raw - x_mean) / x_std

        for n_units in hidden_size_options:
            F_train = build_features(X_train, n_units)
            F_blank_test = build_features(X_blank_test, n_units)
            F_sample = build_features(X_sample, n_units)
            X_flat = F_train.reshape(-1, F_train.shape[-1])
            y_flat = y_train.reshape(-1)

            weights = np.zeros(X_flat.shape[1])
            learning_rate = 0.075 / np.sqrt(n_units)
            losses = []
            snapshots = {}

            for epoch in range(1, max_epochs + 1):
                pred_flat = X_flat @ weights
                residual = pred_flat - y_flat
                losses.append(float(np.mean(residual ** 2)))
                grad = 2.0 * (X_flat.T @ residual) / len(y_flat)
                weights -= learning_rate * grad

                if epoch in epoch_options:
                    train_pred_norm = (F_train.reshape(-1, F_train.shape[-1]) @ weights).reshape(y_train.shape)
                    blank_test_pred_norm = (F_blank_test.reshape(-1, F_blank_test.shape[-1]) @ weights).reshape(X_blank_test_raw.shape)
                    sample_pred_norm = (F_sample.reshape(-1, F_sample.shape[-1]) @ weights).reshape(X_sample_raw.shape)
                    train_pred = train_pred_norm * y_std + y_mean
                    blank_test_pred = blank_test_pred_norm * y_std + y_mean
                    sample_pred = sample_pred_norm * y_std + y_mean
                    train_rmse = float(np.sqrt(np.mean((train_pred - y_train_raw) ** 2)))
                    blank_test_rmse = float(np.sqrt(np.mean((blank_test_pred - y_blank_test_raw) ** 2)))
                    sample_rmse = float(np.sqrt(np.mean((sample_pred - y_sample_raw) ** 2)))
                    snapshots[epoch] = (
                        weights.copy(),
                        np.array(losses),
                        train_rmse,
                        blank_test_rmse,
                        sample_rmse,
                        sample_pred,
                    )

            for epoch in epoch_options:
                _, epoch_losses, train_rmse, blank_test_rmse, sample_rmse, sample_pred = snapshots[epoch]
                residual = sample_pred[example_index] - y_sample_raw[example_index]
                key = f"{n_train}|{n_units}|{epoch}"
                results[key] = {
                    "time": time.round(3).tolist(),
                    "signal": X_sample_raw[example_index].round(4).tolist(),
                    "baseline": y_sample_raw[example_index].round(4).tolist(),
                    "prediction": sample_pred[example_index].round(4).tolist(),
                    "residual": residual.round(4).tolist(),
                    "losses": epoch_losses.round(5).tolist(),
                    "train_rmse": round(train_rmse, 3),
                    "blank_test_rmse": round(blank_test_rmse, 3),
                    "test_rmse": round(sample_rmse, 3),
                    "final_loss": round(float(epoch_losses[-1]), 4),
                    "training_size": int(n_train),
                    "hidden_size": int(n_units),
                    "epochs": int(epoch),
                    "parameter_count": int(n_units + 2),
                    "training_mode": "blank chromatograms only",
                }

    return {
        "training_size_options": training_size_options,
        "hidden_size_options": hidden_size_options,
        "epoch_options": epoch_options,
        "results": results,
    }


# -----------------------------------------------------------------------------
# 4. Transformer-attention payload: vary attention sharpness and selected token.
# -----------------------------------------------------------------------------

def make_attention_dashboard_payload():
    focus_options = [1.0, 2.0, 3.0, 5.0]
    short_tokens = ["gradient", "pH", "temp.", "flow", "pressure", "resolution", "time"]
    results = {}
    for focus in focus_options:
        weights = attention_matrix(EMBEDDINGS, focus_boost=focus)
        results[f"{focus:.1f}"] = {
            "weights": weights.round(4).tolist(),
            "row_entropy": [round(float(-np.sum(row * np.log(row + 1e-12))), 3) for row in weights],
        }
    return {
        "tokens": TOKENS,
        "short_tokens": short_tokens,
        "focus_options": focus_options,
        "results": results,
    }


explorer_payload = {
    "palette": {
        "cyan": IDEAS_CYAN,
        "magenta": IDEAS_MAGENTA,
        "gold": IDEAS_GOLD,
        "green": IDEAS_GREEN,
        "text": IDEAS_TEXT,
        "grid": IDEAS_GRID,
        "pale": IDEAS_PALE_CYAN,
    },
    "nn": make_nn_dashboard_payload(),
    "q": make_q_dashboard_payload(),
    "rnn": make_rnn_dashboard_payload(),
    "attention": make_attention_dashboard_payload(),
}

payload_json = json.dumps(explorer_payload)
container_id = "ideas_ilm_explorer_" + uuid.uuid4().hex[:8]

html_template = r'''
<div id="__CID__" class="ideas-ml-explorer">
  <style>
    #__CID__ {
      font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
      color: #16495A;
      background: #ffffff;
      border: 1px solid #D9F7FF;
      border-radius: 14px;
      padding: 16px;
      max-width: 100%;
      box-sizing: border-box;
      overflow: visible !important;
    }
    #__CID__ * { box-sizing: border-box; }
    #__CID__ h3 { margin: 0 0 4px 0; }
    #__CID__ .subtitle { font-size: 0.92rem; margin-bottom: 12px; }
    #__CID__ .controls {
      display: grid;
      grid-template-columns: repeat(auto-fit, minmax(185px, 1fr));
      gap: 10px;
      margin-bottom: 12px;
    }
    #__CID__ label { font-weight: 600; font-size: 0.90rem; display: block; margin-bottom: 4px; }
    #__CID__ select, #__CID__ textarea {
      width: 100%;
      border: 1px solid #BDEFFF;
      border-radius: 8px;
      padding: 6px 8px;
      color: #16495A;
      background: white;
    }
    #__CID__ .layout {
      display: grid;
      grid-template-columns: minmax(360px, 1.35fr) minmax(270px, 0.85fr);
      gap: 16px;
      align-items: start;
    }
    @media (max-width: 850px) {
      #__CID__ .layout { grid-template-columns: 1fr; }
    }
    #__CID__ canvas {
      width: 100%;
      max-width: 860px;
      height: auto;
      border: 1px solid #D9F7FF;
      border-radius: 10px;
      background: #fbfeff;
      display: block;
    }
    #__CID__ .card {
      border: 1px solid #D9F7FF;
      background: #F8FCFE;
      border-radius: 12px;
      padding: 12px;
      margin-bottom: 10px;
    }
    #__CID__ .metric {
      display: grid;
      grid-template-columns: 1fr auto;
      gap: 8px;
      border-bottom: 1px solid #E5F8FD;
      padding: 5px 0;
      font-size: 0.94rem;
    }
    #__CID__ .metric:last-child { border-bottom: none; }
    #__CID__ textarea { min-height: 78px; margin-top: 6px; resize: vertical; }
    #__CID__ button {
      border: 1px solid #43CEFF;
      background: white;
      color: #16495A;
      border-radius: 8px;
      padding: 7px 10px;
      cursor: pointer;
      margin-right: 6px;
      margin-top: 6px;
    }
    #__CID__ button:hover { background: #D9F7FF; }
    #__CID__ .note { font-size: 0.90rem; line-height: 1.35; }
    #__CID__ .controlGroup { display: none; }
    #__CID__ .suggested { display: none; border-left: 4px solid #FFC000; padding: 9px 11px; background: #FFF2BF; border-radius: 9px; margin-top: 8px; font-size: 0.92rem; }
    #__CID__ .suggested.visible { display: block; }
    #__CID__ .suggested ol { padding-left: 1.2rem; }
    #__CID__ .suggested li { margin-bottom: 0.6rem; }
  </style>

  <h3>Intelligent Machine Learning — interactive exercise</h3>
  <div class="subtitle">Dr. Tijmen S. Bos / Innovative Data Evaluation And Separations (IDEAS) — bos-ideas.com</div>

  <div class="controls">
    <div>
      <label for="__CID___module">Module</label>
      <select id="__CID___module">
        <option value="nn">Neural network response surface</option>
        <option value="q">Q-learning policy</option>
        <option value="rnn">RNN memory</option>
        <option value="attention">Transformer attention</option>
      </select>
    </div>

    <div class="controlGroup nnControl">
      <label for="__CID___nn_arch">Network architecture</label>
      <select id="__CID___nn_arch"></select>
    </div>
    <div class="controlGroup nnControl">
      <label for="__CID___train_size">Training-set size</label>
      <select id="__CID___train_size"></select>
    </div>
    <div class="controlGroup nnControl">
      <label for="__CID___nn_view">Neural-network view</label>
      <select id="__CID___nn_view">
        <option value="actual">True synthetic response</option>
        <option value="predicted">Neural-network prediction</option>
        <option value="error">Prediction error</option>
      </select>
    </div>

    <div class="controlGroup qControl">
      <label for="__CID___episodes">Q-learning episodes</label>
      <select id="__CID___episodes"></select>
    </div>
    <div class="controlGroup qControl">
      <label for="__CID___epsilon">Exploration rate ε</label>
      <select id="__CID___epsilon"></select>
    </div>
    <div class="controlGroup qControl">
      <label for="__CID___q_view">Q-learning view</label>
      <select id="__CID___q_view">
        <option value="policy">Policy arrows</option>
        <option value="value">Value map</option>
        <option value="learning">Learning curve</option>
      </select>
    </div>

    <div class="controlGroup rnnControl">
      <label for="__CID___rnn_train_size">RNN blank-training-set size</label>
      <select id="__CID___rnn_train_size"></select>
    </div>
    <div class="controlGroup rnnControl">
      <label for="__CID___rnn_hidden">RNN hidden units</label>
      <select id="__CID___rnn_hidden"></select>
    </div>
    <div class="controlGroup rnnControl">
      <label for="__CID___rnn_epochs">RNN training epochs</label>
      <select id="__CID___rnn_epochs"></select>
    </div>
    <div class="controlGroup rnnControl">
      <label for="__CID___rnn_view">RNN view</label>
      <select id="__CID___rnn_view">
        <option value="prediction">Baseline prediction</option>
        <option value="error">Prediction error</option>
        <option value="loss">Training curve</option>
      </select>
    </div>

    <div class="controlGroup attentionControl">
      <label for="__CID___focus">Attention sharpness</label>
      <select id="__CID___focus"></select>
    </div>
    <div class="controlGroup attentionControl">
      <label for="__CID___token">Selected query token</label>
      <select id="__CID___token"></select>
    </div>
  </div>

  <div class="layout">
    <div><canvas id="__CID___canvas" width="860" height="580"></canvas></div>
    <div>
      <div class="card" id="__CID___info"></div>
      <div class="card">
        <strong>Course questions</strong>
        <div class="note" id="__CID___questions"></div>
        <textarea id="__CID___answer1" placeholder="Answer 1"></textarea>
        <textarea id="__CID___answer2" placeholder="Answer 2"></textarea>
        <button id="__CID___showAnswers">Show suggested answers</button>
        <button id="__CID___useAnswers">Use suggested answers in boxes</button>
        <button id="__CID___copy">Copy answers</button>
        <button id="__CID___download">Download answers</button>
        <div class="note suggested" id="__CID___suggestedAnswers"></div>
      </div>
    </div>
  </div>
</div>

<script>
(function() {
  const root = document.getElementById("__CID__");
  const payload = __PAYLOAD__;
  const palette = payload.palette;
  const canvas = document.getElementById("__CID___canvas");
  const ctx = canvas.getContext("2d");
  const moduleControl = document.getElementById("__CID___module");
  const nnArch = document.getElementById("__CID___nn_arch");
  const trainSize = document.getElementById("__CID___train_size");
  const nnView = document.getElementById("__CID___nn_view");
  const episodesControl = document.getElementById("__CID___episodes");
  const epsilonControl = document.getElementById("__CID___epsilon");
  const qView = document.getElementById("__CID___q_view");
  const rnnTrainSizeControl = document.getElementById("__CID___rnn_train_size");
  const rnnHiddenControl = document.getElementById("__CID___rnn_hidden");
  const rnnEpochsControl = document.getElementById("__CID___rnn_epochs");
  const rnnViewControl = document.getElementById("__CID___rnn_view");
  const focusControl = document.getElementById("__CID___focus");
  const tokenControl = document.getElementById("__CID___token");
  const info = document.getElementById("__CID___info");
  const questions = document.getElementById("__CID___questions");
  const suggestedAnswersBox = document.getElementById("__CID___suggestedAnswers");
  const answer1Box = document.getElementById("__CID___answer1");
  const answer2Box = document.getElementById("__CID___answer2");

  function addOptions(select, values, labels) {
    select.innerHTML = "";
    values.forEach(function(value, i) {
      const option = document.createElement("option");
      option.value = String(value);
      option.textContent = labels ? labels[i] : String(value);
      select.appendChild(option);
    });
  }

  addOptions(nnArch, payload.nn.architectures.map(function(x) { return x.key; }), payload.nn.architectures.map(function(x) { return x.label; }));
  addOptions(trainSize, payload.nn.training_sizes, payload.nn.training_sizes.map(function(x) { return String(x) + " examples"; }));
  addOptions(episodesControl, payload.q.episode_options, payload.q.episode_options.map(function(x) { return String(x) + " episodes"; }));
  addOptions(epsilonControl, payload.q.epsilon_options, payload.q.epsilon_options.map(function(x) { return "ε = " + Number(x).toFixed(2); }));
  addOptions(rnnTrainSizeControl, payload.rnn.training_size_options, payload.rnn.training_size_options.map(function(x) { return String(x) + " sequences"; }));
  addOptions(rnnHiddenControl, payload.rnn.hidden_size_options, payload.rnn.hidden_size_options.map(function(x) { return String(x) + " hidden unit" + (Number(x) === 1 ? "" : "s"); }));
  addOptions(rnnEpochsControl, payload.rnn.epoch_options, payload.rnn.epoch_options.map(function(x) { return String(x) + " epochs"; }));
  addOptions(focusControl, payload.attention.focus_options, payload.attention.focus_options.map(function(x) { return "sharpness = " + Number(x).toFixed(1); }));
  addOptions(tokenControl, payload.attention.tokens.map(function(_, i) { return i; }), payload.attention.tokens);

  trainSize.value = "140";
  episodesControl.value = "150";
  epsilonControl.value = "0.25";
  rnnTrainSizeControl.value = "30";
  rnnHiddenControl.value = "3";
  rnnEpochsControl.value = "30";
  focusControl.value = "3";
  tokenControl.value = String(payload.attention.tokens.indexOf("resolution"));

  function expandOutputArea() {
    const safeSelectors = [".jp-OutputArea-output", ".jp-RenderedHTMLCommon", ".jp-OutputArea-child", ".jp-Cell-outputWrapper", ".jp-Cell-outputArea", ".output_html", ".output_subarea"];
    const stopSelectors = [".jp-Notebook", ".jp-NotebookPanel-notebook", ".jp-WindowedPanel-viewport", ".jp-WindowedPanel-outer", ".jp-MainAreaWidget"];
    function expandElement(el) {
      if (!el || !el.style) return;
      el.style.setProperty("height", "auto", "important");
      el.style.setProperty("max-height", "none", "important");
      el.style.setProperty("overflow", "visible", "important");
      el.style.setProperty("overflow-y", "visible", "important");
      el.style.setProperty("contain", "none", "important");
    }
    safeSelectors.forEach(function(sel) { expandElement(root.closest(sel)); });
    let node = root.parentElement;
    for (let i = 0; i < 18 && node; i++) {
      if (stopSelectors.some(function(sel) { return node.matches && node.matches(sel); })) break;
      if (node === document.body || node === document.documentElement) break;
      const className = String(node.className || "");
      const looksLikeOutputWrapper = className.includes("Output") || className.includes("output") || className.includes("RenderedHTML") || className.includes("Cell-output") || className.includes("lm-Widget");
      if (looksLikeOutputWrapper) expandElement(node);
      node = node.parentElement;
    }
  }

  function clearCanvas() {
    ctx.clearRect(0, 0, canvas.width, canvas.height);
    ctx.fillStyle = "#ffffff";
    ctx.fillRect(0, 0, canvas.width, canvas.height);
    ctx.textAlign = "left";
    ctx.textBaseline = "alphabetic";
  }
  function hexToRgb(hex) {
    const clean = hex.replace("#", "");
    return [parseInt(clean.substring(0, 2), 16), parseInt(clean.substring(2, 4), 16), parseInt(clean.substring(4, 6), 16)];
  }
  function mix(c1, c2, t) {
    const a = hexToRgb(c1), b = hexToRgb(c2);
    return "rgb(" + Math.round(a[0] + (b[0] - a[0]) * t) + "," + Math.round(a[1] + (b[1] - a[1]) * t) + "," + Math.round(a[2] + (b[2] - a[2]) * t) + ")";
  }
  function colourScale(value, min, max, mode) {
    let t = (value - min) / Math.max(max - min, 1e-9);
    t = Math.max(0, Math.min(1, t));
    if (mode === "diverging") {
      if (t < 0.5) return mix(palette.magenta, "#ffffff", t / 0.5);
      return mix("#ffffff", palette.gold, (t - 0.5) / 0.5);
    }
    if (t < 0.5) return mix("#ffffff", palette.cyan, t / 0.5);
    return mix(palette.cyan, palette.magenta, (t - 0.5) / 0.5);
  }
  function matrixMinMax(matrix) {
    let min = Infinity, max = -Infinity;
    matrix.forEach(function(row) { row.forEach(function(v) { if (v < min) min = v; if (v > max) max = v; }); });
    return {min: min, max: max};
  }
  function drawAxes(x0, y0, w, h, xlabel, ylabel) {
    ctx.strokeStyle = palette.text;
    ctx.lineWidth = 1;
    ctx.beginPath();
    ctx.moveTo(x0, y0 + h);
    ctx.lineTo(x0 + w, y0 + h);
    ctx.moveTo(x0, y0);
    ctx.lineTo(x0, y0 + h);
    ctx.stroke();
    ctx.fillStyle = palette.text;
    ctx.font = "13px sans-serif";
    ctx.textAlign = "center";
    ctx.fillText(xlabel, x0 + w / 2, y0 + h + 42);
    ctx.save();
    ctx.translate(24, y0 + h / 2);
    ctx.rotate(-Math.PI / 2);
    ctx.fillText(ylabel, 0, 0);
    ctx.restore();
  }
  function drawHeatmap(matrix, xValues, yValues, title, xlabel, ylabel, options) {
    options = options || {};
    clearCanvas();
    const x0 = 76, y0 = 48, w = 620, h = 420;
    const rows = matrix.length, cols = matrix[0].length;
    const mm = matrixMinMax(matrix);
    const min = options.min !== undefined ? options.min : mm.min;
    const max = options.max !== undefined ? options.max : mm.max;
    const mode = options.mode || "sequential";
    for (let r = 0; r < rows; r++) {
      for (let c = 0; c < cols; c++) {
        ctx.fillStyle = colourScale(matrix[r][c], min, max, mode);
        ctx.fillRect(x0 + c * w / cols, y0 + (rows - 1 - r) * h / rows, w / cols + 0.5, h / rows + 0.5);
      }
    }
    drawAxes(x0, y0, w, h, xlabel, ylabel);
    ctx.fillStyle = palette.text;
    ctx.font = "18px sans-serif";
    ctx.textAlign = "left";
    ctx.fillText(title, x0, 28);
    ctx.font = "12px sans-serif";
    ctx.textAlign = "center";
    [0, 0.25, 0.5, 0.75, 1].forEach(function(t) {
      const x = x0 + t * w;
      const index = Math.round(t * (xValues.length - 1));
      ctx.fillText(String(xValues[index]), x, y0 + h + 18);
    });
    ctx.textAlign = "right";
    [0, 0.25, 0.5, 0.75, 1].forEach(function(t) {
      const y = y0 + h - t * h;
      const index = Math.round(t * (yValues.length - 1));
      ctx.fillText(String(yValues[index]), x0 - 8, y + 4);
    });
    const lx = 725, ly = 76, lw = 24, lh = 240;
    for (let i = 0; i < lh; i++) {
      const value = max - (i / (lh - 1)) * (max - min);
      ctx.fillStyle = colourScale(value, min, max, mode);
      ctx.fillRect(lx, ly + i, lw, 1);
    }
    ctx.strokeStyle = palette.text;
    ctx.strokeRect(lx, ly, lw, lh);
    ctx.fillStyle = palette.text;
    ctx.textAlign = "left";
    ctx.fillText(max.toFixed(1), lx + 34, ly + 5);
    ctx.fillText(min.toFixed(1), lx + 34, ly + lh);
  }

  function drawLineChart(series, smooth, title, xlabel, ylabel) {
    clearCanvas();
    const x0 = 72, y0 = 50, w = 680, h = 410;
    const all = series.concat(smooth || []);
    const ymin = Math.min.apply(null, all), ymax = Math.max.apply(null, all);
    drawAxes(x0, y0, w, h, xlabel, ylabel);
    ctx.fillStyle = palette.text;
    ctx.font = "18px sans-serif";
    ctx.textAlign = "left";
    ctx.fillText(title, x0, 28);
    function draw(arr, color, width) {
      ctx.strokeStyle = color;
      ctx.lineWidth = width;
      ctx.beginPath();
      arr.forEach(function(v, i) {
        const x = x0 + (i / Math.max(arr.length - 1, 1)) * w;
        const y = y0 + h - ((v - ymin) / Math.max(ymax - ymin, 1e-9)) * h;
        if (i === 0) ctx.moveTo(x, y); else ctx.lineTo(x, y);
      });
      ctx.stroke();
    }
    draw(series, palette.magenta, 1.2);
    if (smooth) draw(smooth, palette.cyan, 2.4);
  }

  function nnKey() { return nnArch.value + "|" + trainSize.value; }
  function qKey() { return episodesControl.value + "|" + Number(epsilonControl.value).toFixed(2); }
  function rnnKey() { return rnnTrainSizeControl.value + "|" + rnnHiddenControl.value + "|" + rnnEpochsControl.value; }

  function drawNN() {
    const result = payload.nn.results[nnKey()];
    const view = nnView.value;
    const title = view === "actual" ? "True synthetic chromatographic response" : view === "predicted" ? "Neural-network prediction" : "Prediction error (predicted − true)";
    const options = view === "error" ? {min: -22, max: 22, mode: "diverging"} : {min: 0, max: 100};
    drawHeatmap(result[view], result.gradient, result.pH, title, "Gradient time (min)", "pH", options);
    info.innerHTML = "<strong>Neural network</strong>" +
      "<div class='metric'><span>Architecture</span><span>" + result.architecture_label + "</span></div>" +
      "<div class='metric'><span>Training examples</span><span>" + result.training_size + "</span></div>" +
      "<div class='metric'><span>Parameters</span><span>" + result.n_parameters + "</span></div>" +
      "<div class='metric'><span>Training RMSE</span><span>" + result.train_rmse + "</span></div>" +
      "<div class='metric'><span>Independent-test RMSE</span><span>" + result.test_rmse + "</span></div>" +
      "<p class='note'>Changing the training-set size and architecture changes how well the response surface is learned. A larger model is not automatically better when the data are sparse.</p>";
    questions.innerHTML = "<ol><li>Does more training data improve the error map?</li><li>When does increasing the number of units or layers help, and when does it not?</li></ol>";
  }

  function drawQ() {
    const result = payload.q.results[qKey()];
    const view = qView.value;
    if (view === "learning") {
      drawLineChart(result.returns, result.rolling, "Q-learning reward during training", "Episode", "Total reward");
    } else {
      const matrix = view === "value" ? result.value : payload.q.reward;
      drawHeatmap(matrix, payload.q.gradient, payload.q.pH, view === "value" ? "Value map after Q-learning" : "Learned policy arrows over reward map", "Gradient time (min)", "pH");
      if (view === "policy") {
        const x0 = 76, y0 = 48, w = 620, h = 420;
        const rows = result.policy.length, cols = result.policy[0].length;
        const arrows = ["↑", "→", "↓", "←"];
        ctx.fillStyle = palette.text;
        ctx.font = "bold 21px sans-serif";
        ctx.textAlign = "center";
        for (let r = 0; r < rows; r++) {
          for (let c = 0; c < cols; c++) {
            const x = x0 + (c + 0.5) * w / cols;
            const y = y0 + (rows - 1 - r + 0.57) * h / rows;
            ctx.fillText(arrows[result.policy[r][c]], x, y);
          }
        }
      }
    }
    info.innerHTML = "<strong>Q-learning</strong>" +
      "<div class='metric'><span>Episodes</span><span>" + episodesControl.value + "</span></div>" +
      "<div class='metric'><span>Exploration ε</span><span>" + Number(epsilonControl.value).toFixed(2) + "</span></div>" +
      "<div class='metric'><span>Last reward avg.</span><span>" + result.last20 + "</span></div>" +
      "<div class='metric'><span>Maximum value</span><span>" + result.max_value + "</span></div>" +
      "<p class='note'>Episodes control the amount of learning. Exploration controls how often the agent tries non-greedy actions. Too little exploration may miss promising regions; too much exploration slows exploitation.</p>";
    questions.innerHTML = "<ol><li>How does the policy change when the number of episodes is low?</li><li>What happens when exploration is very low or very high?</li></ol>";
  }

  function drawRNN() {
    clearCanvas();
    const data = payload.rnn.results[rnnKey()];
    const view = rnnViewControl.value;

    if (view === "loss") {
      drawLineChart(data.losses, null, "RNN training curve", "Epoch", "Mean squared error (normalized)");
    } else {
      const x0 = 72, y0 = 50, w = 680, h = 410;
      const time = data.time;
      const signal = data.signal, baseline = data.baseline, prediction = data.prediction, residual = data.residual;
      let all = [];
      if (view === "prediction") all = signal.concat(baseline).concat(prediction);
      if (view === "error") all = residual.concat([0]);
      const ymin = Math.min.apply(null, all), ymax = Math.max.apply(null, all);
      const sx = function(t) { return x0 + (t - time[0]) / (time[time.length - 1] - time[0]) * w; };
      const sy = function(v) { return y0 + h - (v - ymin) / Math.max(ymax - ymin, 1e-9) * h; };
      const ylabel = view === "error" ? "Prediction error (a.u.)" : "Detector response (a.u.)";
      drawAxes(x0, y0, w, h, "Time (min)", ylabel);
      function line(arr, color, width, dash) {
        ctx.save();
        ctx.strokeStyle = color; ctx.lineWidth = width;
        if (dash) ctx.setLineDash(dash);
        ctx.beginPath();
        arr.forEach(function(v, i) { const x = sx(time[i]), y = sy(v); if (i === 0) ctx.moveTo(x, y); else ctx.lineTo(x, y); });
        ctx.stroke();
        ctx.restore();
      }
      ctx.fillStyle = palette.text;
      ctx.font = "18px sans-serif";
      ctx.textAlign = "left";
      ctx.fillText(view === "error" ? "RNN baseline-prediction error on sample" : "Blank-trained RNN baseline prediction", x0, 28);
      if (view === "prediction") {
        line(signal, palette.cyan, 1.4);
        line(baseline, palette.gold, 2.1);
        line(prediction, palette.magenta, 2.0, [6, 4]);
        const legend = [["Observed sample chromatogram", palette.cyan], ["True synthetic baseline", palette.gold], ["RNN prediction", palette.magenta]];
        ctx.font = "13px sans-serif";
        legend.forEach(function(d, i) { ctx.fillStyle = d[1]; ctx.fillRect(560, 58 + i * 24, 18, 4); ctx.fillStyle = palette.text; ctx.fillText(d[0], 585, 64 + i * 24); });
      } else {
        const zeroY = sy(0);
        ctx.strokeStyle = palette.text;
        ctx.lineWidth = 1;
        ctx.setLineDash([4, 4]);
        ctx.beginPath(); ctx.moveTo(x0, zeroY); ctx.lineTo(x0 + w, zeroY); ctx.stroke();
        ctx.setLineDash([]);
        line(residual, palette.green, 2.0);
        ctx.font = "13px sans-serif";
        ctx.fillStyle = palette.green; ctx.fillRect(560, 58, 18, 4); ctx.fillStyle = palette.text; ctx.fillText("Prediction − true baseline", 585, 64);
      }
    }

    info.innerHTML = "<strong>RNN training from blanks</strong>" +
      "<div class='metric'><span>Blank training sequences</span><span>" + data.training_size + "</span></div>" +
      "<div class='metric'><span>Hidden units</span><span>" + data.hidden_size + "</span></div>" +
      "<div class='metric'><span>Training epochs</span><span>" + data.epochs + "</span></div>" +
      "<div class='metric'><span>Train RMSE on blanks</span><span>" + data.train_rmse + " a.u.</span></div>" +
      "<div class='metric'><span>Blank-test RMSE</span><span>" + data.blank_test_rmse + " a.u.</span></div>" +
      "<div class='metric'><span>Peaked-sample RMSE</span><span>" + data.test_rmse + " a.u.</span></div>" +
      "<div class='metric'><span>Parameters</span><span>" + data.parameter_count + "</span></div>" +
      "<p class='note'>This panel uses actually trained recurrent models. Training uses blank chromatograms only: baseline drift plus detector noise, without analyte peaks. The displayed chromatogram is a sample with peaks, so students can see what happens when a blank-trained baseline model is applied to a real sample-like trace. Section 5 above shows the full Elman-RNN training version.</p>";
    questions.innerHTML = "<ol><li>What happens when the number of blank training sequences increases?</li><li>Does a blank-trained model ignore peaks perfectly, or can peaks still affect the predicted baseline?</li></ol>";
  }

  function drawAttention() {
    clearCanvas();
    const focusKey = Number(focusControl.value).toFixed(1);
    const result = payload.attention.results[focusKey];
    const weights = result.weights;
    const tokens = payload.attention.short_tokens;
    const fullTokens = payload.attention.tokens;
    const selected = Number(tokenControl.value);
    const n = tokens.length;
    const x0 = 142, y0 = 54, size = 340;
    const cell = size / n;
    const max = Math.max(0.35, Math.max.apply(null, weights.flat()));
    ctx.fillStyle = palette.text;
    ctx.font = "18px sans-serif";
    ctx.textAlign = "left";
    ctx.fillText("Transformer-style self-attention", x0, 28);

    for (let r = 0; r < n; r++) {
      for (let c = 0; c < n; c++) {
        const value = weights[r][c];
        ctx.fillStyle = colourScale(value, 0, max);
        ctx.fillRect(x0 + c * cell, y0 + r * cell, cell + 0.5, cell + 0.5);
        ctx.strokeStyle = "rgba(255,255,255,0.75)";
        ctx.strokeRect(x0 + c * cell, y0 + r * cell, cell, cell);
        if (value >= 0.09) {
          ctx.fillStyle = palette.text;
          ctx.font = "10px sans-serif";
          ctx.textAlign = "center";
          ctx.textBaseline = "middle";
          ctx.fillText(Math.round(value * 100) + "%", x0 + (c + 0.5) * cell, y0 + (r + 0.5) * cell);
        }
      }
    }
    ctx.strokeStyle = palette.magenta;
    ctx.lineWidth = 3;
    ctx.strokeRect(x0, y0 + selected * cell, size, cell);
    ctx.lineWidth = 1;
    ctx.strokeStyle = palette.text;
    ctx.strokeRect(x0, y0, size, size);

    ctx.font = "12px sans-serif";
    ctx.fillStyle = palette.text;
    ctx.textAlign = "right";
    ctx.textBaseline = "middle";
    tokens.forEach(function(tok, i) { ctx.fillText(tok, x0 - 9, y0 + (i + 0.5) * cell); });
    ctx.textAlign = "left";
    tokens.forEach(function(tok, i) {
      const x = x0 + (i + 0.5) * cell;
      const y = y0 + size + 12;
      ctx.save(); ctx.translate(x, y); ctx.rotate(-0.70); ctx.fillText(tok, 0, 0); ctx.restore();
    });
    ctx.font = "13px sans-serif";
    ctx.textAlign = "center";
    ctx.fillText("Attended token", x0 + size / 2, y0 + size + 105);
    ctx.save(); ctx.translate(26, y0 + size / 2); ctx.rotate(-Math.PI / 2); ctx.fillText("Query token", 0, 0); ctx.restore();

    const bx = 560, by = 78, bw = 225, bh = 270;
    ctx.fillStyle = palette.text;
    ctx.textAlign = "left";
    ctx.font = "14px sans-serif";
    ctx.fillText("Selected query: " + fullTokens[selected], bx, by - 20);
    const row = weights[selected];
    for (let i = 0; i < n; i++) {
      const barW = row[i] / Math.max.apply(null, row) * bw;
      const y = by + i * 34;
      ctx.fillStyle = i === selected ? palette.gold : palette.cyan;
      ctx.fillRect(bx, y, barW, 18);
      ctx.strokeStyle = "#ffffff";
      ctx.strokeRect(bx, y, barW, 18);
      ctx.fillStyle = palette.text;
      ctx.font = "12px sans-serif";
      ctx.fillText(tokens[i] + "  " + Math.round(row[i] * 100) + "%", bx + 4, y + 14);
    }

    info.innerHTML = "<strong>Transformer attention</strong>" +
      "<div class='metric'><span>Attention sharpness</span><span>" + Number(focusControl.value).toFixed(1) + "</span></div>" +
      "<div class='metric'><span>Selected query</span><span>" + fullTokens[selected] + "</span></div>" +
      "<div class='metric'><span>Row entropy</span><span>" + result.row_entropy[selected] + "</span></div>" +
      "<p class='note'>Higher sharpness concentrates attention on fewer tokens. Lower entropy means the selected token relies on a smaller number of context tokens.</p>";
    questions.innerHTML = "<ol><li>Which tokens receive the most attention from the selected query?</li><li>What changes when attention sharpness increases?</li></ol>";
  }


  function escapeHTML(value) {
    return String(value).replace(/[&<>"]/g, function(ch) {
      return {"&": "&amp;", "<": "&lt;", ">": "&gt;", "\"": "&quot;"}[ch];
    });
  }

  function currentSuggestedAnswers() {
    if (moduleControl.value === "nn") {
      const result = payload.nn.results[nnKey()];
      return [
        `More training data generally reduces the error map and improves the independent-test RMSE, provided the model has enough capacity. In this setting the selected model has ${result.n_parameters} parameters, a training RMSE of ${result.train_rmse}, and a test RMSE of ${result.test_rmse}.`,
        "Increasing the number of units or layers helps only when the model is underfitting and enough training examples are available. With sparse data, a larger network can fit the training data without improving the response surface between measured points."
      ];
    }
    if (moduleControl.value === "q") {
      const result = payload.q.results[qKey()];
      return [
        `With few episodes, the policy is usually incomplete or locally biased. More episodes allow the value estimates and policy arrows to stabilize; here the final rolling reward is ${result.last20}.`,
        "Very low exploration can trap the agent in a poor part of the method-development grid. Very high exploration keeps trying random actions for too long, which slows exploitation of good pH/gradient-time regions."
      ];
    }
    if (moduleControl.value === "rnn") {
      const data = payload.rnn.results[rnnKey()];
      return [
        `Increasing the number of blank training sequences usually improves the learned baseline model because it sees more baseline-drift and detector-noise variation. In this setting the blank-test RMSE is ${data.blank_test_rmse} a.u. after ${data.epochs} epochs.`,
        "A blank-trained RNN should learn the background trend rather than analyte peaks, because peaks are absent from the training blanks. It may still be influenced by large or broad peaks when applied to sample chromatograms, so the prediction should be checked against blank and quality-control runs."
      ];
    }
    const result = payload.attention.results[Number(focusControl.value).toFixed(1)];
    const selected = Number(tokenControl.value);
    const row = result.weights[selected];
    const tokens = payload.attention.tokens;
    const ranked = row.map(function(v, i) { return {token: tokens[i], value: v}; }).sort(function(a, b) { return b.value - a.value; }).slice(0, 3);
    return [
      `For the selected query token “${tokens[selected]}”, the largest attention weights go to ${ranked.map(function(x) { return x.token + " (" + Math.round(x.value * 100) + "% )"; }).join(", ")}.`,
      `Increasing attention sharpness concentrates the row on fewer tokens and lowers the attention entropy. In this setting the selected row entropy is ${result.row_entropy[selected]}, so the query is using a relatively ${result.row_entropy[selected] < 1.4 ? "focused" : "distributed"} context.`
    ];
  }

  function updateSuggestedAnswers() {
    const a = currentSuggestedAnswers();
    suggestedAnswersBox.innerHTML = "<strong>Suggested answers</strong><ol><li>" + escapeHTML(a[0]) + "</li><li>" + escapeHTML(a[1]) + "</li></ol>";
  }

  function fillSuggestedAnswers() {
    const a = currentSuggestedAnswers();
    answer1Box.value = a[0];
    answer2Box.value = a[1];
  }

  function updateControls() {
    const mod = moduleControl.value;
    root.querySelectorAll(".controlGroup").forEach(function(el) { el.style.display = "none"; });
    root.querySelectorAll("." + mod + "Control").forEach(function(el) { el.style.display = "block"; });
  }
  function render() {
    expandOutputArea();
    updateControls();
    if (moduleControl.value === "nn") drawNN();
    if (moduleControl.value === "q") drawQ();
    if (moduleControl.value === "rnn") drawRNN();
    if (moduleControl.value === "attention") drawAttention();
    updateSuggestedAnswers();
    expandOutputArea();
  }

  [moduleControl, nnArch, trainSize, nnView, episodesControl, epsilonControl, qView, rnnTrainSizeControl, rnnHiddenControl, rnnEpochsControl, rnnViewControl, focusControl, tokenControl].forEach(function(el) {
    el.addEventListener("change", render);
  });

  document.getElementById("__CID___showAnswers").addEventListener("click", function() {
    updateSuggestedAnswers();
    suggestedAnswersBox.classList.toggle("visible");
    document.getElementById("__CID___showAnswers").textContent = suggestedAnswersBox.classList.contains("visible") ? "Hide suggested answers" : "Show suggested answers";
    expandOutputArea();
  });
  document.getElementById("__CID___useAnswers").addEventListener("click", function() {
    if (!confirm("Replace the two answer boxes with the current suggested answers?")) return;
    fillSuggestedAnswers();
  });

  function collectAnswers() {
    return "HPLC 2026 — Intelligent Machine Learning exercise\n" +
      "Dr. Tijmen S. Bos / IDEAS — bos-ideas.com\n" +
      "Module: " + moduleControl.value + "\n\n" +
      "Answer 1:\n" + document.getElementById("__CID___answer1").value + "\n\n" +
      "Answer 2:\n" + document.getElementById("__CID___answer2").value + "\n";
  }
  document.getElementById("__CID___copy").addEventListener("click", async function() {
    await navigator.clipboard.writeText(collectAnswers());
  });
  document.getElementById("__CID___download").addEventListener("click", function() {
    const blob = new Blob([collectAnswers()], {type: "text/plain"});
    const url = URL.createObjectURL(blob);
    const a = document.createElement("a");
    a.href = url;
    a.download = "hplc2026_intelligent_ml_answers.txt";
    a.click();
    URL.revokeObjectURL(url);
  });

  expandOutputArea();
  setTimeout(expandOutputArea, 50);
  setTimeout(expandOutputArea, 400);
  if (window.ResizeObserver) {
    const resizeObserver = new ResizeObserver(function() { expandOutputArea(); });
    resizeObserver.observe(root);
  }
  render();
})();
</script>
'''

html = html_template.replace('__CID__', container_id).replace('__PAYLOAD__', payload_json)
display(HTML(html))
print("Dynamic interactive exercise ready.")


## 8. Take-home interpretation

The examples above describe different types of “intelligence” in machine learning:

- A **neural network** learns an input–output mapping from examples.
- **Q-learning** learns actions from reward feedback.
- An **actor–critic** architecture separates action selection from value estimation.
- An **RNN** uses memory to process sequential data.
- A **transformer** uses attention to weight relevant context directly.

For liquid-phase separations, these concepts are most useful when they are connected to a clear experimental question: prediction, classification, optimization, method control, or signal interpretation.
